# 🚀 AI Video Dubbing Studio - Cloud GPU Server (Google Colab / Kaggle)

Welcome to the **AI Video Dubbing Cloud GPU Server**! This notebook runs the full heavy processing pipeline (Faster-Whisper, Chatterbox Multilingual / XTTS v2 Zero-Shot Voice Cloning, Wav2Lip GAN, UVR Background Separator) on a **free Cloud GPU (NVIDIA Tesla T4 / A100 / P100)**.

### 📋 Setup (always run this first):
1. In Google Colab menu, go to **Runtime > Change runtime type** and select **T4 GPU** (or A100 if Colab Pro).
2. Run **Cell 1** (Install Dependencies & Download Models). Only needs re-running if the runtime restarts.

### Then pick ONE of these two ways to dub a video:

**Option A - Everything inside Colab, no local app needed (easiest):**
- Run **Cell 3** below. Upload a video or paste a YouTube link in the form on the right, adjust settings if you want (they default to the highest quality already), and run - it downloads/plays the finished video right here.

**Option B - Use the local Video Dubbing GUI on your own PC:**
- Run **Cell 2** (Start Cloud GPU Server & Tunnel) instead of Cell 3.
- Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`) it prints and paste it into your local **Video Dubbing GUI**'s "Remote Cloud GPU" field.

### 🔗 If a YouTube link fails to download:
Colab's IP is shared by thousands of users, so YouTube sometimes blocks downloads from it with a bot-check. This only matters if you're pasting a YouTube link (not needed for uploads). If it happens:
1. Export a `cookies.txt` from a browser where you're logged into YouTube (e.g. the "Get cookies.txt LOCALLY" extension).
2. Upload it to the Colab file browser on the left (wherever you like), then paste its path (e.g. `/content/cookies.txt`) into the **YouTube Cookies File** field in Cell 3's form.
3. ⚠️ **Treat this file as secret** - it's your live logged-in session, not a reusable API key. Never commit it or upload it to GitHub (the repo's `.gitignore` already blocks any `cookies*.txt` from being committed by accident). It only needs to exist in this Colab runtime, which is wiped when the runtime restarts.

In [1]:
# Cell 1: Check GPU & Install Dependencies
!nvidia-smi

# Always start from /content so re-running this cell (e.g. after a git pull) never
# accidentally "cd's into itself" and lands in a nested/wrong directory.
%cd /content

# Clone repo or pull latest code
!git clone https://github.com/Asadullah404/Ai_Video_Dubbing.git dubbing_app || (cd dubbing_app && git pull)
%cd /content/dubbing_app

# Install dependencies
!pip install -q fastapi uvicorn python-multipart pycloudflared nest-asyncio deep-translator
!pip install -q faster-whisper coqui-tts pyannote.audio audio-separator[gpu] speechbrain groq librosa soundfile noisereduce pedalboard resemblyzer gTTS yt-dlp opencv-python

# The chain of pip installs above can leave NumPy in a corrupted, half-upgraded state on
# disk (a common Colab pitfall - multiple packages each nudging NumPy's version can produce
# a mix of old/new files, causing errors like "cannot import name '_center' from
# numpy._core.umath" the first time something imports it). `--force-reinstall` alone isn't
# enough here - it overwrites files the new version ships but does NOT delete leftover files
# from whatever version was there before, which is exactly how a newer umath.py ends up
# paired with a stale compiled extension that predates it. Uninstall first so no old files
# survive, then do a fresh install.
!pip uninstall -y -q numpy
!pip install -q --no-cache-dir numpy

# Download pre-trained Wav2Lip and S3FD weights
!mkdir -p Wav2Lip/face_detection/detection/sfd
!wget -q -c 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights2/wav2lip_gan.1.1.pth' -O 'Wav2Lip/wav2lip_gan.pth'
!wget -q -c 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights1/s3fd-619a316812.1.1.pth' -O 'Wav2Lip/face_detection/detection/sfd/s3fd.pth'

print("\n✅ All Dependencies and Pre-trained Models are installed and ready!")
print("⚠️ If this is a fresh install (first run), go to Runtime > Restart session now before running Cell 2 or Cell 3, to make sure NumPy loads cleanly.")

Sun Aug  9 12:32:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Cell 2: Launch Cloud GPU Server & Expose via Cloudflare Tunnel
import os

# Optional: Set your HuggingFace token for PyAnnote speaker diarization
os.environ["HF_TOKEN"] = ""
os.environ["COQUI_TOS_AGREED"] = "1"

# Run server
!python colab_server.py

📦 Auto-installing missing dependency: coqui-tts...

🚀 Starting AI Video Dubbing GPU Server...
🎮 GPU Device: Tesla T4 (14.56 GB VRAM)
⚡ CUDA Available: True

🌐 Creating secure public Cloudflare Tunnel...
 * Running on https://opening-receivers-october-television.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics

*****************************************************************
🎉 YOUR CLOUD GPU SERVER IS READY!
🔗 Public URL: https://opening-receivers-october-television.trycloudflare.com
👉 Copy and paste the https://...trycloudflare.com URL into your Local Dubbing GUI.
*****************************************************************

[1581b059] Starting fresh processing (progress reset)
[1581b059] 
[1581b059] Enhanced Video Dubbing System v2.0
[1581b059] ============================================================
[1581b059] Video Duration: 0.3 minutes
[1581b059] Video FPS: 30.00
[1581b059] Processing Chunks: 1
[1581b059] Device: cuda
[1581b059] TTS Engine: C

In [ ]:
#@title Cell 3: 🎬 One-Click Dubbing (upload a video or paste a link, right here in Colab) { display-mode: "form" }
#@markdown Fill in the form on the right, then run this cell (▶). Settings already default to the highest quality - only Target Language really needs changing.

video_source = "Upload from computer" #@param ["Upload from computer", "YouTube URL"]
youtube_url = "" #@param {type:"string"}
youtube_cookies_path = "" #@param {type:"string"}
source_language = "en" #@param ["auto","en","es","fr","de","it","pt","pl","tr","ru","nl","cs","ar","zh-cn","ja","ko","hi","ur","hu"]
target_language = "es" #@param ["en","es","fr","de","it","pt","pl","tr","ru","nl","cs","ar","zh-cn","ja","ko","hi","ur","hu","bn","ta","te","ml","th","vi","id","ms","fa","sw","ne","si"]
whisper_model = "large-v3" #@param ["tiny","base","small","medium","large-v3"]
voice_quality = "ultra" #@param ["standard","high","ultra"]
enable_lipsync = True #@param {type:"boolean"}
preserve_background_audio = True #@param {type:"boolean"}
hf_token = "" #@param {type:"string"}
groq_token = "" #@param {type:"string"}

import os
%cd /content/dubbing_app

# Sanity-check NumPy loads cleanly before anything else. A corrupted/half-upgraded NumPy
# install (common in Colab after several pip installs in Cell 1) fails with a cryptic
# "cannot import name '_center' from numpy._core.umath" deep inside video_dubbing_core's
# import chain - catch it here with an actionable message instead.
try:
    import numpy
    from numpy._core import strings as _np_strings  # exact import chain that breaks when corrupted
    numpy.array([1, 2, 3]).sum()
except Exception as e:
    raise RuntimeError(
        "NumPy failed a basic sanity check (likely corrupted by earlier pip installs): "
        f"{e}\n\n>>> FIX: Go to Runtime > Restart session, then re-run Cell 1 and this cell again. <<<"
    )

if hf_token.strip():
    os.environ["HF_TOKEN"] = hf_token.strip()
if groq_token.strip():
    os.environ["Groq_TOKEN"] = groq_token.strip()
if youtube_cookies_path.strip():
    os.environ["YT_COOKIES_FILE"] = youtube_cookies_path.strip()
os.environ["COQUI_TOS_AGREED"] = "1"

# Make sure Chatterbox Multilingual (primary voice cloning engine) is installed - same
# --no-deps install colab_server.py does, needed here too since this cell calls the
# dubbing pipeline directly instead of going through colab_server.py.
from download_and_setup import step_install_chatterbox
step_install_chatterbox()

# Get the video
if video_source == "Upload from computer":
    from google.colab import files
    print("📤 Choose a video file to upload...")
    uploaded = files.upload()
    if not uploaded:
        raise Exception("No file uploaded.")
    video_path = list(uploaded.keys())[0]
else:
    if not youtube_url.strip():
        raise Exception("Please paste a YouTube URL in the form above.")
    from video_dubbing_core import download_youtube_video
    video_path = download_youtube_video(youtube_url.strip())
    if not video_path:
        raise Exception("YouTube download failed - see the log above.")

print(f"\n🎬 Video ready: {video_path}")

# Run the full dubbing pipeline directly on this GPU
from video_dubbing_core import EnhancedVideoDubbing

dubber = EnhancedVideoDubbing(
    video_path=video_path,
    source_lang=source_language,
    target_lang=target_language,
    whisper_model=whisper_model,
    voice_quality=voice_quality,
    enable_lipsync=enable_lipsync,
    preserve_bg=preserve_background_audio,
    hf_token=os.getenv("HF_TOKEN"),
    groq_token=os.getenv("Groq_TOKEN"),
)
dubber.process()

# Show and download the result
result_path = "results/dubbed_video.mp4"
if os.path.exists(result_path):
    print(f"\n✅ Done! Output: {result_path}")
    from IPython.display import Video, display
    display(Video(result_path, embed=True, width=640))
    from google.colab import files
    files.download(result_path)
else:
    print("❌ Processing finished but the output video wasn't found - check the log above for errors.")